Aqui fazer a exploração inicial do dataset para construir a camada silver

In [5]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]



In [6]:
import os
print(os.environ.get("JAVA_HOME"))

/usr/lib/jvm/java-17-openjdk-amd64


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("cybersecurity_ingestion") \
    .getOrCreate()



df_incedents = spark.read.parquet( "../data/bronze/2026-03-21/incidents_master.parquet")
df_market_impact = spark.read.parquet( "../data/bronze/2026-03-21/market_impact.parquet")
df_financial_impact = spark.read.parquet( "../data/bronze/2026-03-21/financial_impact .parquet")

vou fazer analises gerais em cada dataset e depois paratir para um analise mais especifica de cada 1

In [ ]:

df_incedents.show(5)

26/03/21 13:09:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------+--------------------+-------------------+----------+----------------+------------------+--------------+-----------------+------------+-------------+-----------------------+--------------+---------------+---------------------+-----------------------+--------------------+----------------+----------------------+------------------------+---------+--------------------+--------------+--------------------+---------------------+----------------+---------------+-------------+-------------+-----------+--------------------+--------------------+--------------------+
|  incident_id|        company_name|company_revenue_usd|country_hq|industry_primary|industry_secondary|employee_count|is_public_company|stock_ticker|incident_date|incident_date_estimated|discovery_date|disclosure_date|attack_vector_primary|attack_vector_secondary|        attack_chain|attributed_group|attribution_confidence|data_compromised_records|data_type|    systems_affected|downtime_hours| data_source_primary|data_so

In [14]:
df_financial_impact.show(5)

+-------------+---------------+------------------+-------------------+---------------+-------------+-----------------+--------------+-------------------+--------------------+--------------+-----------------+----------------------+----------------------+----------------------+--------------------+-----+--------------------+--------------------+
|  incident_id|direct_loss_usd|direct_loss_method|ransom_demanded_usd|ransom_paid_usd|ransom_source|recovery_cost_usd|legal_fees_usd|regulatory_fine_usd|insurance_payout_usd|total_loss_usd|total_loss_method|total_loss_lower_bound|total_loss_upper_bound|inflation_adjusted_usd|      cpi_index_used|notes|          created_at|          updated_at|
+-------------+---------------+------------------+-------------------+---------------+-------------+-----------------+--------------+-------------------+--------------------+--------------+-----------------+----------------------+----------------------+----------------------+--------------------+-----+-----

In [15]:
df_market_impact.show(5)

+-------------+------------+---------------+--------------------+--------------+--------------+---------------+-----------------------+---------------------+--------------------+-------------------------+------------------+------------------+-------------------+----------------+----------+-----------+-----------+--------------+----------+---------------+-----------+-------------------------------+------------------------+-----------------------+---------------------------+----------------------------+----------------------+-----+--------------------+--------------------+
|  incident_id|stock_ticker|price_7d_before|price_disclosure_day|price_1d_after|price_7d_after|price_30d_after|volume_avg_30d_baseline|volume_disclosure_day|        sector_index|sector_return_same_period|abnormal_return_1d|abnormal_return_7d|abnormal_return_30d|car_neg1_to_pos1|car_0_to_7|car_0_to_30|car_0_to_90|t_statistic_1d|p_value_1d|t_statistic_30d|p_value_30d|earnings_announcement_within_7d|market_cap_at_disclosu

In [ ]:
df_incedents.select("country_hq").distinct().show()

+----------+
|country_hq|
+----------+
|        FI|
|        NL|
|        PL|
|        MX|
|        AT|
|        CZ|
|        PT|
|        HK|
|        TW|
|        CL|
|        AU|
|        CA|
|        GB|
|        BR|
|        DE|
|        ES|
|        IL|
|        ZA|
|        KR|
|        US|
+----------+
only showing top 20 rows


In [ ]:
print("Colunas do DataFrame:")
print(df_incedents.columns)

print("\nSchema do DataFrame:")
df_incedents.printSchema()

print("\nContagem de valores nulos por coluna:")
from pyspark.sql.functions import count, when, isnull
df_incedents.select([count(when(isnull(c), c)).alias(c) for c in df_incedents.columns]).show()

Schema do DataFrame:
root
 |-- incident_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- company_revenue_usd: double (nullable = true)
 |-- country_hq: string (nullable = true)
 |-- industry_primary: string (nullable = true)
 |-- industry_secondary: string (nullable = true)
 |-- employee_count: long (nullable = true)
 |-- is_public_company: boolean (nullable = true)
 |-- stock_ticker: string (nullable = true)
 |-- incident_date: string (nullable = true)
 |-- incident_date_estimated: boolean (nullable = true)
 |-- discovery_date: string (nullable = true)
 |-- disclosure_date: string (nullable = true)
 |-- attack_vector_primary: string (nullable = true)
 |-- attack_vector_secondary: string (nullable = true)
 |-- attack_chain: string (nullable = true)
 |-- attributed_group: string (nullable = true)
 |-- attribution_confidence: string (nullable = true)
 |-- data_compromised_records: double (nullable = true)
 |-- data_type: string (nullable = true)
 |-- systems_

26/03/21 15:39:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------+-------------------+----------+----------------+------------------+--------------+-----------------+------------+-------------+-----------------------+--------------+---------------+---------------------+-----------------------+------------+----------------+----------------------+------------------------+---------+----------------+--------------+-------------------+---------------------+----------------+---------------+-------------+-------------+-----------+-----+----------+----------+
|incident_id|company_name|company_revenue_usd|country_hq|industry_primary|industry_secondary|employee_count|is_public_company|stock_ticker|incident_date|incident_date_estimated|discovery_date|disclosure_date|attack_vector_primary|attack_vector_secondary|attack_chain|attributed_group|attribution_confidence|data_compromised_records|data_type|systems_affected|downtime_hours|data_source_primary|data_source_secondary|data_source_type|confidence_tier|quality_score|quality_grade|revie

In [ ]:
print("colunas do dataFrame:")
print(df_market_impact.columns)

print("\nSchema do dataFrame:")
df_market_impact.printSchema()

print("\ncontagem de valores nulos por coluna:")
from pyspark.sql.functions import count, when, isnull
df_market_impact.select([count(when(isnull(c), c)).alias(c) for c in df_market_impact.columns]).show()

Colunas do DataFrame:
['incident_id', 'stock_ticker', 'price_7d_before', 'price_disclosure_day', 'price_1d_after', 'price_7d_after', 'price_30d_after', 'volume_avg_30d_baseline', 'volume_disclosure_day', 'sector_index', 'sector_return_same_period', 'abnormal_return_1d', 'abnormal_return_7d', 'abnormal_return_30d', 'car_neg1_to_pos1', 'car_0_to_7', 'car_0_to_30', 'car_0_to_90', 't_statistic_1d', 'p_value_1d', 't_statistic_30d', 'p_value_30d', 'earnings_announcement_within_7d', 'market_cap_at_disclosure', 'volume_ratio_disclosure', 'pre_incident_volatility_30d', 'post_incident_volatility_30d', 'days_to_price_recovery', 'notes', 'created_at', 'updated_at']

Schema do DataFrame:
root
 |-- incident_id: string (nullable = true)
 |-- stock_ticker: string (nullable = true)
 |-- price_7d_before: double (nullable = true)
 |-- price_disclosure_day: double (nullable = true)
 |-- price_1d_after: double (nullable = true)
 |-- price_7d_after: double (nullable = true)
 |-- price_30d_after: double (nul

In [16]:
print("Colunas do DataFrame:")
print(df_financial_impact.columns)

print("\nSchema do DataFrame:")
df_financial_impact.printSchema()

print("\nContagem de valores nulos por coluna:")
from pyspark.sql.functions import count, when, isnull
df_financial_impact.select([count(when(isnull(c), c)).alias(c) for c in df_financial_impact.columns]).show()

Colunas do DataFrame:
['incident_id', 'direct_loss_usd', 'direct_loss_method', 'ransom_demanded_usd', 'ransom_paid_usd', 'ransom_source', 'recovery_cost_usd', 'legal_fees_usd', 'regulatory_fine_usd', 'insurance_payout_usd', 'total_loss_usd', 'total_loss_method', 'total_loss_lower_bound', 'total_loss_upper_bound', 'inflation_adjusted_usd', 'cpi_index_used', 'notes', 'created_at', 'updated_at']

Schema do DataFrame:
root
 |-- incident_id: string (nullable = true)
 |-- direct_loss_usd: double (nullable = true)
 |-- direct_loss_method: string (nullable = true)
 |-- ransom_demanded_usd: double (nullable = true)
 |-- ransom_paid_usd: double (nullable = true)
 |-- ransom_source: string (nullable = true)
 |-- recovery_cost_usd: double (nullable = true)
 |-- legal_fees_usd: double (nullable = true)
 |-- regulatory_fine_usd: double (nullable = true)
 |-- insurance_payout_usd: double (nullable = true)
 |-- total_loss_usd: double (nullable = true)
 |-- total_loss_method: string (nullable = true)
 

Oque foi feito 
-Script de ingestão 
-contagem de nulos 
-estrutura inicial
Para fazer
-padronização dos nomes 
-verificar duplicadas e outliers de acordo com cada regra de negocios(por exmemplo se tiver idade negativa tira , agr temperatura deixa )